In [1]:

!pip -q install requests beautifulsoup4 langchain-text-splitters chromadb sentence-transformers groq


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/

In [2]:

import requests
from bs4 import BeautifulSoup
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from groq import Groq

print("All required libraries imported successfully!")


All required libraries imported successfully!


In [3]:

import requests
from bs4 import BeautifulSoup

url = "https://www.w3schools.com/python/python_intro.asp"

response = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=20
)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

# Extract the main tutorial content
content = soup.find("div", id="main")

if content is None:
    raise ValueError("Could not find tutorial content on the page.")

# Remove navigation and unrelated elements
for element in content.select(
    "script, style, nav, .w3-sidebar, #footer, .w3-info"
):
    element.decompose()

scraped_text = content.get_text(separator="\n", strip=True)

print("Scraping successful!")
print("Characters collected:", len(scraped_text))
print("\nPreview:\n")
print(scraped_text[:3000])


Scraping successful!
Characters collected: 2121

Preview:

Python
Introduction
❮ Previous
Next ❯
Share
Link Copied
What is Python?
Python is a popular programming language. It was created by Guido van Rossum, 
and released in 1991.
It is used for:
web development (server-side),
software development,
mathematics,
system scripting.
What can Python do?
Python can be used on a server to create web applications.
Python can be used alongside software to create workflows.
Python can connect to database systems. It can also read and modify files.
Python can be used to handle big data and perform complex mathematics.
Python can be used for rapid prototyping, or for production-ready software development.
Why Python?
Python works on different platforms (Windows, Mac, Linux, Raspberry Pi, etc).
Python has a simple syntax similar to the English language.
Python has syntax that allows developers to write programs with fewer lines than some other programming languages.
Python runs on an interpreter s

In [4]:

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80
)

chunks = text_splitter.split_text(scraped_text)

print("Total chunks created:", len(chunks))
print("\nFirst chunk:\n")
print(chunks[0])


Total chunks created: 5

First chunk:

Python
Introduction
❮ Previous
Next ❯
Share
Link Copied
What is Python?
Python is a popular programming language. It was created by Guido van Rossum, 
and released in 1991.
It is used for:
web development (server-side),
software development,
mathematics,
system scripting.
What can Python do?
Python can be used on a server to create web applications.
Python can be used alongside software to create workflows.
Python can connect to database systems. It can also read and modify files.


In [5]:

import chromadb
from sentence_transformers import SentenceTransformer

# Load an embedding model
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# Create a vector database
client = chromadb.Client()

collection = client.get_or_create_collection(
    name="eduguide_python"
)

# Convert chunks into embeddings
embeddings = embedding_model.encode(chunks).tolist()

# Store chunks and embeddings
collection.upsert(
    ids=[f"python_{i}" for i in range(len(chunks))],
    documents=chunks,
    embeddings=embeddings,
    metadatas=[
        {
            "source": "W3Schools Python Introduction",
            "url": "https://www.w3schools.com/python/python_intro.asp"
        }
        for _ in chunks
    ]
)

print("Successfully stored", collection.count(), "chunks in ChromaDB!")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Successfully stored 5 chunks in ChromaDB!


In [6]:

question = "What is Python and what is it used for?"

# Convert the question into an embedding
question_embedding = embedding_model.encode(
    [question]
).tolist()

# Search for the most relevant chunks
results = collection.query(
    query_embeddings=question_embedding,
    n_results=3
)

print("Student question:", question)
print("\nRetrieved information:\n")

for i, document in enumerate(results["documents"][0], start=1):
    print(f"--- Result {i} ---")
    print(document)
    print()


Student question: What is Python and what is it used for?

Retrieved information:

--- Result 1 ---
Python
Introduction
❮ Previous
Next ❯
Share
Link Copied
What is Python?
Python is a popular programming language. It was created by Guido van Rossum, 
and released in 1991.
It is used for:
web development (server-side),
software development,
mathematics,
system scripting.
What can Python do?
Python can be used on a server to create web applications.
Python can be used alongside software to create workflows.
Python can connect to database systems. It can also read and modify files.

--- Result 2 ---
Python can connect to database systems. It can also read and modify files.
Python can be used to handle big data and perform complex mathematics.
Python can be used for rapid prototyping, or for production-ready software development.
Why Python?
Python works on different platforms (Windows, Mac, Linux, Raspberry Pi, etc).
Python has a simple syntax similar to the English language.
Python has s

In [7]:

from google.colab import userdata
from groq import Groq

# Read the secret named "groq"
api_key = userdata.get("groq")

groq_client = Groq(api_key=api_key)

print("Groq connected successfully!")


Groq connected successfully!


In [9]:

def ask_eduguide(question):
    # Convert the question into an embedding
    question_embedding = embedding_model.encode(
        [question]
    ).tolist()

    # Retrieve relevant educational content
    results = collection.query(
        query_embeddings=question_embedding,
        n_results=3
    )

    context = "\n\n".join(results["documents"][0])

    # Ask Groq to answer using the retrieved content
    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are EduGuide AI, a friendly educational tutor. "
                    "Answer using the provided W3Schools context. "
                    "Explain concepts simply for beginners. "
                    "If the context does not contain the answer, "
                    "say that the available tutorial content does not "
                    "provide enough information. Do not invent facts."
                )
            },
            {
                "role": "user",
                "content": (
                    f"Educational context:\n{context}\n\n"
                    f"Student question: {question}"
                )
            }
        ],
        temperature=0.2
    )

    answer = response.choices[0].message.content

    return answer


# Test EduGuide AI
question = "What is Python and why is it useful?"
print("Student:", question)
print("\nEduGuide AI:", ask_eduguide(question))


Student: What is Python and why is it useful?

EduGuide AI: **What is Python?**  
Python is a popular, high‑level programming language that was created by Guido van Rossum and first released in 1991. It is designed to be easy to read and write, with a syntax that feels close to plain English and is influenced by mathematics.

**Why is Python useful?**  
According to the tutorial content, Python is useful for a wide range of tasks:

| Use‑case | Why it matters |
|----------|----------------|
| **Web development (server‑side)** | You can build dynamic web applications that run on a server. |
| **Software development** | Python can be used to create both small scripts and large, production‑ready applications. |
| **Mathematics & data science** | It handles big data and complex mathematical calculations efficiently. |
| **System scripting** | Python can read, write, and modify files, and interact with databases, making it great for automating workflows. |
| **Rapid prototyping** | Because 

In [10]:

def ask_eduguide(question):
    question_embedding = embedding_model.encode(
        [question]
    ).tolist()

    results = collection.query(
        query_embeddings=question_embedding,
        n_results=3
    )

    context = "\n\n".join(results["documents"][0])

    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are EduGuide AI, a friendly educational tutor. "
                    "Answer using the supplied W3Schools context. "
                    "Explain concepts simply. If the context is insufficient, "
                    "say so instead of inventing an answer."
                )
            },
            {
                "role": "user",
                "content": (
                    f"Context:\n{context}\n\n"
                    f"Student question: {question}"
                )
            }
        ],
        temperature=0.2
    )

    answer = response.choices[0].message.content

    sources = sorted({
        metadata["url"]
        for metadata in results["metadatas"][0]
        if metadata and "url" in metadata
    })

    return answer, sources


question = "What is Python and why is it useful?"
answer, sources = ask_eduguide(question)

print("EduGuide AI:\n")
print(answer)
print("\nSources:")
for source in sources:
    print(source)


EduGuide AI:

**What is Python?**  
Python is a popular programming language that was created by Guido van Rossum and first released in 1991. It is a general‑purpose language, meaning you can use it for many different kinds of software.

**Why is Python useful?**  
- **Versatile applications** – Python can be used for web development (server‑side), software development, mathematics, system scripting, big‑data processing, and more.  
- **Cross‑platform** – It runs on Windows, macOS, Linux, Raspberry Pi, and many other operating systems.  
- **Simple, readable syntax** – Python’s code looks almost like plain English, so you can write programs with fewer lines than many other languages.  
- **Rapid prototyping** – Because of its ease of use, you can quickly build and test ideas, then turn them into production‑ready software.  
- **Rich ecosystem** – Python can connect to databases, read and modify files, and integrate with other software, making it a powerful tool for building workflows a

In [11]:
print("Collection name:", collection.name)
print("Number of chunks:", collection.count())

Collection name: eduguide_python
Number of chunks: 5


In [13]:

import json
from google.colab import files

# Read all chunks from your existing collection
data = collection.get(
    include=["documents", "metadatas"]
)

export_chunks = []

for chunk_id, document, metadata in zip(
    data["ids"],
    data["documents"],
    data["metadatas"]
):
    export_chunks.append({
        "id": chunk_id,
        "content": document,
        "url": (
            metadata.get("url")
            or metadata.get("source")
            or "https://www.w3schools.com/python/"
        ),
        "title": metadata.get("title", "W3Schools Python Tutorial"),
        "metadata": metadata
    })

# Save chunks and metadata to JSON
filename = "w3schools_chunks.json"

with open(filename, "w", encoding="utf-8") as f:
    json.dump(export_chunks, f, ensure_ascii=False, indent=2)

print("Exported chunks:", len(export_chunks))
print("File created:", filename)

# Download the JSON file to your laptop
files.download(filename)


Exported chunks: 5
File created: w3schools_chunks.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>